# Preliminaries: Functions for Exact Diagonalization

In [ ]:
using LinearAlgebra
using SparseArrays #Sparse arrays are arrays that contain enough zeros that storing them in a special data structure leads to savings in space and execution time, compared to dense arrays.

#Sparse Matrix representations for local operators (definition as sparse guarantes that Kron() returns also sparse):
σx = sparse([0 1; 1 0])
σy = sparse([0 -1im; 1im 0])
σz = sparse([1 0; 0 -1])
Identity = sparse([1 0; 0 1]) #or Using LinearAlgebra and Matrix{Float64}(I, 2, 2)
σplus = (σx +1im*σy)/2
σminus = adjoint(σplus)

"""
Builds O_j as I⊗...⊗I⊗O⊗I...⊗I for a system of N sites.

Note: If the local representation is an sparse matrix, the output of this function will also be an sparse matrix. This is very important for larger N.

**Parameters:**
* `N (Int64):` System size.
* `j (Int64):` Site where the local operator O is acting.
* `matrix (Matrix{Any}):` Matrix representation for the local operator O. 

**Return:**
* `M (Matrix{Any}):` Global operator of O_j i.e. Tensor product I⊗...⊗I⊗O⊗I...⊗I.
"""
function Enlarge_Matrix_site_j(N::Int64, j::Int64, matrix)

    global Identity
    
    M = Identity
    j == 1 ? M = matrix : nothing
    
    for i=2:N 
        i == j ? M = sparse(kron(matrix, M)) :  M = sparse(kron(Identity, M)) #It is more efficient to save everything as sparse matrices. 
    end

    return M
end

"""
Builds A_iB_j as I⊗...⊗I⊗B⊗I...⊗I⊗A⊗I⊗I...⊗I (for i<j) for a system of N sites.

Note: If the local representation are sparse matrices, the output of this function will also be an sparse matrix. This is very important for larger N.

**Parameters:**
* `N (Int64):` System size.
* `i (Int64):` Site where the local operator A is acting.
* `j (Int64):` Site where the local operator B is acting.
* `matrix_i (Int64):` Matrix representation for the local operator A.
* `matrix_j (Int64):` Matrix representation for the local operator B.

**Return:**
* `M (Matrix{Any}):` Global operator of A_iB_j i.e. Tensor product I⊗...⊗I⊗B⊗I...⊗I⊗A⊗I⊗I...⊗I (for i<j).
"""
function Enlarge_Matrix_i_Matrix_j(N::Int64, i::Int64, j::Int64, matrix_i, matrix_j)
    global Identity
    
    M = Identity

    j == 1 ? M = matrix_j : nothing
    i == 1 ? M = matrix_i : nothing
 
    for k=2:N 
        if k == j
            M = sparse(kron(matrix_j, M))
        elseif k == i
            M = sparse(kron(matrix_i, M))
        else
            M = sparse(kron(Identity, M))     
        end
    end

    return M
end

"""
Builds the creation operator C†_j as an sparse matrix based on the JW transformation C†_j = (∑_i<j σz_i) σ-_j.

**Parameters:**
* `N (Int64):` System size.
* `j (Int64):` Site where creation operator is acting.

**Return:**
* `Matrix (SparseMatrixCSC{ComplexF64, Int64}):` Matrix representation of C†_j.
"""
function Build_Cdag_Matrix(N::Int64, j::Int64)
    global σminus, σz, Identity

    if j == 1
        Matrix = σminus
    else
        Matrix = σz
    end

    for i=2:N 
        if i <= j-1
        Matrix = sparse(kron(σz, Matrix))
        elseif i == j
        Matrix = sparse(kron(σminus, Matrix))
        else
        Matrix = sparse(kron(Identity, Matrix))  
        end
    end 

    return Matrix

end

"""
Builds the annihilation operator C_j as an sparse matrix based on the JW transformation C_j = (∑_i<j σz_i) σ+_j.

**Parameters:**
* `N (Int64):` System size.
* `j (Int64):` Site where annihilation operator is acting.

**Return:**
* `Matrix (SparseMatrixCSC{ComplexF64, Int64}):` Matrix representation of C_j.
"""
function Build_C_Matrix(N::Int64, j::Int64)
    global σplus, σz, Identity

    if j == 1
        Matrix = σplus
    else
        Matrix = σz
    end
    
    for i=2:N 
        if i <= j-1
        Matrix = sparse(kron(σz, Matrix))
        elseif i == j
        Matrix = sparse(kron(σplus, Matrix))
        else
        Matrix = sparse(kron(Identity, Matrix))
        end
    end 

    return Matrix

end

"""
Builds a hopping operator C†_iC_j as an sparse matrix based on the JW transformation. 

**Parameters:**
* `N (Int64):` System size.
* `i (Int64):` Site where creation operator is acting.
* `j (Int64):` Site where annihilation operator is acting.

**Return:**
* `Matrix (SparseMatrixCSC{ComplexF64, Int64}):` Matrix representation of C_j.
"""
function Build_C_dagi_Cj_Matrix(N::Int64, i::Int64, j::Int64)
    #In this case some JW strings cancel between them. C†_iC_j = σ-_i (Π_i<k<j σz_k) σ+_j (this works exactly the same for i<j and j<i because σ-*σz=σ- and σz*σ+=σ+).
    
    global σminus, σplus, σz, Identity

    if i == j #Particle number operator i.e. C†_iC_i = σ-_i σ+_i
        return Enlarge_Matrix_site_j(N, j, σminus*σplus)
    end  
    
    if i > j #C†_iC_j = (C†_jC_i)†
        return adjoint(Build_C_dagi_Cj_Matrix(N, j, i))
    end

    Matrix = Identity
    i == 1 ? Matrix = σminus : nothing
 
    for k=2:N # i < j case
        if k == i
            Matrix = sparse(kron(σminus, Matrix))
        elseif k == j
            Matrix = sparse(kron(σplus, Matrix))
        elseif k > i && k < j
            Matrix = sparse(kron(σz, Matrix))       
        else
            Matrix = sparse(kron(Identity, Matrix))      
        end
    end

    return Matrix
end

# Hamiltonian and Lindbladian construction:

In [ ]:
function Build_H(N, Δ, μ, E, J)
    #J: Hopping amplitude.
    #Δ: Nearest-neighbor density-density interaction.
    #μ: Chemical potential of the diode.
    #E: Tilt strength.

    H = zeros(2^N, 2^N)
    
    for i=1:N-1 #H_S
        H += 0.5*J*Build_C_dagi_Cj_Matrix(N, i, i+1)
        H += 0.5*J*Build_C_dagi_Cj_Matrix(N, i+1, i)
        H += Δ*(Build_C_dagi_Cj_Matrix(N, i, i)-0.5*Enlarge_Matrix_site_j(N, 1, Identity))*(Build_C_dagi_Cj_Matrix(N, i+1, i+1)-0.5*Enlarge_Matrix_site_j(N, 1, Identity))
    end

    for i=1:N
        H += (μ + 0.5*E*i)*(Build_C_dagi_Cj_Matrix(N, i, i)-0.5*Enlarge_Matrix_site_j(N, 1, Identity))
    end    
    return H
end

function custom_log(x, base::Real)
    return log(x)/log(base)
end

function Build_Liouvillian_Superoperator(H, L_k_list, local_dimension)

    #l_k has inside the sqrt(γ_K)

    Identity = Diagonal(ones(local_dimension))
    N = custom_log(size(H)[1], local_dimension)    
    superIdentity = Diagonal(ones(Int64(local_dimension^N)))
    
    superH = -1im*(kron(superIdentity, H) - kron(transpose(H), superIdentity))
    superL = spzeros(ComplexF64, size(superH))
    
    for k=1:length(L_k_list)
        L_k = L_k_list[k]
        superL = superL + kron(conj(L_k), L_k) -0.5*(kron(superIdentity, adjoint(L_k)*L_k) + kron(transpose(L_k)*conj(L_k), superIdentity)) 
    end

    Liouvillian_Superoperator = superH + superL
    return Liouvillian_Superoperator*1im #I added this 1im in order to have Schrodinger-type equation
end

# Starting point:

We have a strongly interacting tilted system

$$H = \sum_{i=1}^{D-1} \frac{J}{2}(c_{i}^{\dagger}c_{i+1} + h.c) + \Delta \left( \hat{n}_{j} - \frac{1}{2}\right) \left( \hat{n}_{j+1} - \frac{1}{2}\right) + \sum_{i = 1}^{D} \left( \mu + \frac{iE}{2} \right) \left( \hat{n}_{j} - \frac{1}{2}\right)$$

In [ ]:
global ε_t_array, ts, U

J = 1.0
Δ = 5.0
D = 4
E = 3.65

f_S = [0.0 for j=1:D]
γ_S = [0.0 for j=1:D]

f_S[1] = 1.0
f_S[D] = 0.0
γ_S[1] = 1.0
γ_S[D] = 0.0

L_k_list = []

for i = 1:D
    γ = γ_S[i]
    f = f_S[i]
    push!(L_k_list, sqrt(γ*(1-f))*Build_C_Matrix(D, i))
    push!(L_k_list, sqrt(γ*f)*Build_Cdag_Matrix(D, i))
end

In [ ]:
H = Build_H(D, Δ, μ, E, J)
L = Build_Liouvillian_Superoperator(H, L_k_list, 2)

The idea of this project is to study different methods for finding the NESS.

In [ ]:
I_matrix = Matrix(Enlarge_Matrix_site_j(D, 1, Diagonal(ones(2))))
I_vec = Matrix(reshape(I_matrix, (2^(2*D), 1)))

ρ0_vec = reshape(I_matrix, (2^(2*D), 1))
ρ0_vec = ρ0_vec/(adjoint(I_vec)*ρ0_vec)

In [ ]:
dt = 0.05
U_op = exp(-1im*Matrix(L*dt)

Numsteps = 2000 #13 seconds / 100 steps.

times = [0.0]
ρ_vec = ρ0_vec;

for n =1:Numsteps 
    t = times[end]
    ρ_vec = U_op*ρ_vec
    ρ_vec = ρ_vec/(adjoint(I_vec)*ρ_vec)
    #Here you need to implement the calculation of different observables. Consider currents through each site and the occupation of each site.
    
    push!(times, t+dt)
end

During this project you should study the following questions:

- How do you determine if you have convergence in your quantities? Compare with the NESs solved by diagonalizing L.
- Implement Rungekkuta and Taylor expansion of L to perform the time evolution for a very long time (do it without destroying the sparse structure of L).
- Compare the previous methos with the exact solution. Check running time, errors, impact of dt.
- How is changing the sparse strucutre of $|rho \rangle$ during the time evolution.
- Repear this but now using the ML framework for the non-interacting (compare with analytical solution) and the interacting case (compare with tensor networks and ED). This task sounds hard but do not worry, the main change in your code will be using other function for L (copy+paste). If this works, study the maxima number of leads modes that you can reach for D = 4.